In [ ]:
# %%
print("Hello PPO RLHF")

In [ ]:
# %%
%pip install accelerate datasets evaluate rouge_score bert_score sacrebleu -q
%pip install transformers==4.45.2 trl==0.11.3 -q

In [ ]:
# %%
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)
from trl import (
    PPOConfig,
    PPOTrainer,
    AutoModelForCausalLMWithValueHead,
    SFTTrainer,
    SFTConfig
)
import evaluate

In [ ]:
# %%
# ================= CONFIG =================

MODEL_NAME = "gpt2"
DATASET_NAME = "HumanLLMs/Human-Like-DPO-Dataset"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 4
EPOCHS = 1
LEARNING_RATE = 5e-6
MAX_LENGTH = 512
MAX_RESPONSE_LENGTH = 128

EXP_NAME = "ppo_human_alignment"

BASE_DIR = f"runs/{EXP_NAME}"
CKPT_DIR = f"{BASE_DIR}/checkpoints"
LOG_DIR = f"{BASE_DIR}/logs"
PLOT_DIR = f"{BASE_DIR}/plots"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

In [ ]:
# %%
# ================= DATA =================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset(DATASET_NAME, split="train").train_test_split(test_size=0.2)

train_dataset = dataset["train"].select(range(min(len(dataset["train"]), 4096)))
eval_dataset = dataset["test"].select(range(min(len(dataset["test"]), 1024)))

def format_for_rlhf(example):
    return {
        "query": example["prompt"]
    }

train_dataset = train_dataset.map(format_for_rlhf)
eval_dataset = eval_dataset.map(format_for_rlhf)

In [ ]:
# %%
# ================= SFT =================

sft_config = SFTConfig(
    output_dir=f"{CKPT_DIR}/sft",
    dataset_text_field="chosen",
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    num_train_epochs=1,
    logging_steps=10,
    max_seq_length=512,
    fp16=torch.cuda.is_available()
)

sft_trainer = SFTTrainer(
    model=MODEL_NAME,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config
)

print("Training SFT...")
sft_trainer.train()
sft_trainer.save_model(f"{CKPT_DIR}/sft_model")

In [ ]:
def tokenize_pair(example):
    chosen = tokenizer(example["chosen"], truncation=True, padding="max_length", max_length=512)
    rejected = tokenizer(example["rejected"], truncation=True, padding="max_length", max_length=512)

    return {
        "input_ids_chosen": chosen["input_ids"],
        "mask_chosen": chosen["attention_mask"],
        "input_ids_rejected": rejected["input_ids"],
        "mask_rejected": rejected["attention_mask"]
    }

def reward_loss(model, inputs):
    chosen_rewards = model(
        input_ids=torch.tensor(inputs["input_ids_chosen"]).to(DEVICE),
        attention_mask=torch.tensor(inputs["mask_chosen"]).to(DEVICE)
    ).logits

    rejected_rewards = model(
        input_ids=torch.tensor(inputs["input_ids_rejected"]).to(DEVICE),
        attention_mask=torch.tensor(inputs["mask_rejected"]).to(DEVICE)
    ).logits

    return -torch.nn.functional.logsigmoid(chosen_rewards - rejected_rewards).mean()

In [ ]:
# %%
# ================= REWARD MODEL =================

reward_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1
)

reward_model.config.pad_token_id = tokenizer.eos_token_id

reward_train = train_dataset.map(tokenize_pair, remove_columns=train_dataset.column_names)

optimizer = torch.optim.Adam(reward_model.parameters(), lr=1e-5)

print("Training Reward Model...")
reward_model.to(DEVICE)

for epoch in range(1):
    for sample in reward_train:
        loss = reward_loss(reward_model, sample)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

reward_model.save_pretrained(f"{CKPT_DIR}/reward_model")

In [ ]:
# %%
# ================= PPO =================

policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(f"{CKPT_DIR}/sft_model")
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(f"{CKPT_DIR}/sft_model")

reward_pipe = pipeline(
    "sentiment-analysis",
    model=f"{CKPT_DIR}/reward_model",
    device=0 if torch.cuda.is_available() else -1
)

ppo_config = PPOConfig(
    learning_rate=LEARNING_RATE,
    batch_size=32,
    mini_batch_size=BATCH_SIZE,
    ppo_epochs=4,
    target_kl=0.1,
    gradient_accumulation_steps=1,
    max_grad_norm=1.0
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=train_dataset
)

print("Starting PPO Training...")

for step, batch in enumerate(ppo_trainer.dataloader):

    query_tensors = batch["input_ids"]

    response_tensors = ppo_trainer.generate(
        query_tensors,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.7
    )

    responses = [tokenizer.decode(r) for r in response_tensors]

    texts = [q + r for q, r in zip(batch["query"], responses)]
    rewards = [torch.tensor(o["score"]) for o in reward_pipe(texts)]

    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
    ppo_trainer.log_stats(stats, batch, rewards)

    if step % 10 == 0:
        print(f"Step {step} | KL {stats['objective/kl']:.4f}")


In [ ]:
# %%
# ================= GENERATIVE METRICS =================

bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
bert_metric = evaluate.load("bertscore")

def compute_generative_metrics(model):

    preds = []
    refs = []

    for sample in eval_dataset:

        inputs = tokenizer(sample["prompt"], return_tensors="pt").to(DEVICE)

        out = model.generate(**inputs, max_new_tokens=64)

        pred = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(pred)
        refs.append(sample["chosen"])

    bleu = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    rouge = rouge_metric.compute(predictions=preds, references=refs)
    bert = bert_metric.compute(predictions=preds, references=refs, lang="en")

    return {
        "bleu": bleu["score"],
        "rougeL": rouge["rougeL"],
        "bert_f1": np.mean(bert["f1"])
    }

metrics = compute_generative_metrics(policy_model)

print(metrics)


In [ ]:
# %%
# ================= PLOT =================

history = pd.DataFrame(ppo_trainer.state.log_history)

plt.figure(figsize=(10,5))
loss_logs = history[history["loss"].notna()]
plt.plot(loss_logs["step"], loss_logs["loss"])
plt.title("PPO Loss")
plt.savefig(f"{PLOT_DIR}/ppo_loss.png")
plt.show()

print("Finished PPO RLHF experiment.")
